# Assignment 3. Knowledge Distillation

> ### 이 노트북 읽는 법
> 이 파일은 `3. Knowledge Distillation_answer.ipynb`에 **파이썬 초보자용 해설**을 덧붙인 버전입니다.
> - 원본의 셀(설명·코드·실행 결과)은 **그대로** 두었습니다.
> - 번호가 붙은 코드 셀 **바로 아래**마다 `📘 셀 N 해설` 칸을 추가했습니다. 맨 앞 **Environment Setup 부분(셀 1 ~ 5)은 해설을 생략**했고, 본문인 셀 6 ~ 26을 설명합니다.
> - 해설은 **한 줄 요약 → 코드 뜯어보기 → 출력 해석** 순서입니다. 코드와 실행 결과를 먼저 훑어본 뒤 해설을 읽으면 이해하기 쉽습니다.
>
> **Setup에서 만들어 두고 뒤에서 계속 쓰는 것들**
> - `train_dataset`, `test_loader` : CIFAR-10 학습 데이터 / 테스트 데이터를 128장씩 꺼내 주는 도구
> - `get_train_loader(train_dataset)` : **항상 같은 순서로** 섞이도록 seed를 고정한 학습용 데이터 로더를 만들어 주는 함수 (실험끼리 공정하게 비교하기 위함)
> - `set_seed()` : 난수를 고정하는 함수
> - `state_dict` : 미리 학습된 Teacher 모델의 가중치
> - `VGGCifar9` : **Teacher** 모델 설계도 (합성곱 층 8개) / `VGGCifar5` : **Student** 모델 설계도 (합성곱 층 4개)
>
> **미리 알아 둘 용어**
> - **Knowledge Distillation(지식 증류, KD)** : 크고 정확한 모델(**Teacher**)의 지식을 작고 가벼운 모델(**Student**)에게 가르치는 학습 방법
> - **logits** : 모델이 마지막에 내놓는 클래스별 점수 (확률로 바꾸기 전의 값)
> - **softmax** : 점수들을 합이 1인 **확률**로 바꾸는 함수
> - **loss(손실)** : 모델의 답이 목표와 얼마나 다른지를 나타내는 숫자. 학습은 이 값을 줄이는 과정입니다.
> - **에폭(epoch)** : 학습 데이터 전체를 한 바퀴 도는 것

## Goals

이 실습의 목적은 **Knowledge Distillation**을 활용하여, 작은 모델(Student)이 큰 모델(Teacher)의 지식을 효과적으로 학습하는 방법을 이해하고 실험을 통해 비교하는 것입니다.


## Contents

1. **Baseline 학습 (Cross-Entropy Loss)**
    - Teacher 모델과 Student 모델을 각각 Cross-Entropy Loss만으로 학습시켜 정확도를 비교합니다.
2. **Knowledge Distillation (Soft Targets)**
    - Teacher의 softmax 출력을 활용한 Knowledge Distillation을 적용하고, temperature 및 loss weight에 따른 영향을 분석합니다.
3. **Cosine Loss Minimization (Cosine Loss)**  
    - Teacher와 Student의 convolutional feature를 추출하여, CosineEmbeddingLoss를 적용해 내부 표현 유사도를 증가시키는 방식으로 학습합니다.
4. **Intermediate Regressor (Regressor + MSE)**
    - Teacher의 feature map과 Student의 regressed feature map을 MSE로 정렬하며, 중간 표현을 직접 학습합니다.

# Environment Setup

본 실습에서는 PyTorch와 Torchvision을 활용하여 Knowledge Distillation을 구현합니다. 먼저 필요한 라이브러리를 import하고, 실행 환경(GPU/CPU)을 설정합니다.

## Import Modules
- `torch`, `torch.nn`, `torch.optim`: PyTorch의 핵심 기능 및 신경망, 최적화 알고리즘
- `torchvision.transforms`, `torchvision.datasets`: CIFAR-10 데이터셋 로딩 및 전처리를 위한 모듈
- `collections.OrderedDict`: 이후에 모델 구조 정의 시 순서를 보장하기 위한 dict 구조

In [1]:
# 1
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from collections import OrderedDict

## Data Loading: CIFAR-10

본 실습에서는 CIFAR-10 데이터셋을 사용하여 Knowledge Distillation의 효과를 검증합니다. CIFAR-10은 10개의 클래스로 구성된 32x32 크기의 컬러 이미지 데이터셋입니다.

In [2]:
# 2
# torchvision의 공식 다운로드 서버 에러를 피하기 위해
# Hugging Face Hub에서 CIFAR-10을 로드하는 커스텀 Dataset 클래스
class HFCIFAR10(Dataset):
    def __init__(self, split, transform=None):
        self.data = load_dataset("uoft-cs/cifar10", split=split)
        self.transform = transform
        self.targets = list(self.data["label"])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Subset에서 numpy.int64가 들어올 수 있으므로 Python int로 변환
        idx = int(idx)
        item = self.data[idx]
        # Hugging Face CIFAR-10 field names: img, label
        image = item["img"].convert("RGB")
        label = item["label"]
        if self.transform is not None:
            image = self.transform(image)
        return image, label

# Below we are preprocessing data for CIFAR-10. We use an arbitrary batch size of 128.
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
])

# Loading the CIFAR-10 dataset:
train_dataset = HFCIFAR10("train", transform=transforms_cifar)
test_dataset = HFCIFAR10("test", transform=transforms_cifar)

# Dataloaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

In [3]:
# 3
import random
import numpy as np
import os

def set_seed(seed=44):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# 시드 고정
def get_train_loader(train_dataset, seed=44):
    set_seed(seed)
    def seed_worker(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True,
                                               num_workers=0, generator=g, worker_init_fn=seed_worker)
    return train_loader

## Load Pretrained Model Weights (VGG on CIFAR-10)

Knowledge Distillation에서 중요한 전제는 **강력한 성능을 가진 Teacher 모델**이 존재한다는 것입니다. 본 코드에서는 사전에 학습된 VGG 모델의 가중치를 불러와 Teacher 모델로 사용할 준비를 합니다.

In [4]:
# 4
os.environ["TORCH_HOME"] = "D:\\data"
state_dict_url = "https://github.com/SKKU-ESLAB/pytorch-models/releases/download/samsung/vgg.cifar.pretrained.pth"
state_dict = torch.hub.load_state_dict_from_url(state_dict_url, progress=True)
state_dict = state_dict["state_dict"]

## Define Teacher and Student Models

Knowledge Distillation 실험을 위해 두 개의 모델 구조를 정의합니다. 두 모델은 VGG 스타일의 CNN 구조를 기반으로 하며, **Teacher (VGGCifar9)** 모델은 더 깊고 복잡한 구조, **Student (VGGCifar5)** 모델은 간단한 구조로 설계되어 있습니다.

In [5]:
# 5
class VGGCifar9(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.backbone = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, 64, 3, padding=1, bias=False)),
            ('bn0', nn.BatchNorm2d(64)),
            ('relu0', nn.ReLU(True)),
            ('conv1', nn.Conv2d(64, 128, 3, padding=1, bias=False)),
            ('bn1', nn.BatchNorm2d(128)),
            ('relu1', nn.ReLU(True)),
            ('pool0', nn.MaxPool2d(2)),
            ('conv2', nn.Conv2d(128, 256, 3, padding=1, bias=False)),
            ('bn2', nn.BatchNorm2d(256)),
            ('relu2', nn.ReLU(True)),
            ('conv3', nn.Conv2d(256, 256, 3, padding=1, bias=False)),
            ('bn3', nn.BatchNorm2d(256)),
            ('relu3', nn.ReLU(True)),
            ('pool1', nn.MaxPool2d(2)),
            ('conv4', nn.Conv2d(256, 512, 3, padding=1, bias=False)),
            ('bn4', nn.BatchNorm2d(512)),
            ('relu4', nn.ReLU(True)),
            ('conv5', nn.Conv2d(512, 512, 3, padding=1, bias=False)),
            ('bn5', nn.BatchNorm2d(512)),
            ('relu5', nn.ReLU(True)),
            ('pool2', nn.MaxPool2d(2)),
            ('conv6', nn.Conv2d(512, 512, 3, padding=1, bias=False)),
            ('bn6', nn.BatchNorm2d(512)),
            ('relu6', nn.ReLU(True)),
            ('conv7', nn.Conv2d(512, 512, 3, padding=1, bias=False)),
            ('bn7', nn.BatchNorm2d(512)),
            ('relu7', nn.ReLU(True)),
            ('pool3', nn.MaxPool2d(2)),
        ]))
        self.classifier = nn.Linear(512, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        x = x.mean([2, 3])
        x = self.classifier(x)
        return x

class VGGCifar5(nn.Module):
    def __init__(self) -> None:
        # Generate the same scratch model
        set_seed()
        super().__init__()
        self.backbone = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, 64, 3, padding=1, bias=False)),
            ('bn0', nn.BatchNorm2d(64)),
            ('relu0', nn.ReLU(True)),
            ('pool0', nn.MaxPool2d(2)),
            ('conv1', nn.Conv2d(64, 128, 3, padding=1, bias=False)),
            ('bn1', nn.BatchNorm2d(128)),
            ('relu1', nn.ReLU(True)),
            ('pool1', nn.MaxPool2d(2)),
            ('conv2', nn.Conv2d(128, 256, 3, padding=1, bias=False)),
            ('bn2', nn.BatchNorm2d(256)),
            ('relu2', nn.ReLU(True)),
            ('pool2', nn.MaxPool2d(2)),
            ('conv3', nn.Conv2d(256, 256, 3, padding=1, bias=False)),
            ('bn3', nn.BatchNorm2d(256)),
            ('relu3', nn.ReLU(True)),
            ('pool3', nn.MaxPool2d(2)),
        ]))
        self.classifier = nn.Linear(256, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        x = x.mean([2, 3])
        x = self.classifier(x)
        return x

# 3.1. Baseline 학습 (Cross-Entropy Loss)

## Train & Test Functions

학습 및 검증은 아래 두 함수로 수행되며, Student/Teacher 모델 모두 동일한 루프 구조를 따릅니다.

In [6]:
# 6
def train(model,
          train_loader,
          epochs,
          learning_rate):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs * len(train_loader))

    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            # inputs: A collection of batch_size images
            # labels: A vector of dimensionality batch_size with integers denoting class of each image
            inputs, labels = inputs.cuda(), labels.cuda()

            optimizer.zero_grad()
            outputs = model(inputs)

            # outputs: Output of the network for the collection of images. A tensor of dimensionality batch_size x num_classes
            # labels: The actual labels of the images. Vector of dimensionality batch_size
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

def test(model,
         test_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.cuda(), labels.cuda()

            outputs = model(inputs)
            if isinstance(outputs, tuple):
                outputs = outputs[0]  # for multiple outputs
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

### 📘 셀 6 해설
**한 줄 요약:** 모델을 **학습시키는 함수 `train`** 과 **정확도를 재는 함수 `test`** 를 정의합니다. (정의만 하고 실행은 뒤에서)

**`train(model, train_loader, epochs, learning_rate)`**

준비물 3가지
- `criterion = nn.CrossEntropyLoss()` : 분류 문제의 표준 **손실 함수**. 예측이 정답과 다를수록 큰 값을 냅니다.
- `optimizer = optim.Adam(model.parameters(), lr=learning_rate)` : 가중치를 고쳐 주는 도구(**옵티마이저**). `lr`은 한 번에 고치는 폭(학습률)입니다.
- `scheduler = ...CosineAnnealingLR(optimizer, epochs * len(train_loader))` : 학습률을 처음 값에서 **서서히 0까지** 줄여 가는 계획. `len(train_loader)`는 1 에폭의 묶음 수이므로, 전체 스텝 수를 넘겨준 것입니다.

반복문 (바깥: 에폭 / 안쪽: 묶음)

| 코드 | 의미 |
|---|---|
| `model.train()` | 모델을 **학습 모드**로 전환 |
| `inputs, labels = inputs.cuda(), labels.cuda()` | 이미지와 정답을 GPU로 이동 |
| `optimizer.zero_grad()` | 이전 묶음의 기울기를 0으로 초기화 |
| `outputs = model(inputs)` | 예측값 계산 (묶음 크기 × 10개 클래스 점수) |
| `loss = criterion(outputs, labels)` | 손실 계산 |
| `loss.backward()` | 가중치별 기울기 계산 (역전파) |
| `optimizer.step()` | 가중치 수정 |
| `scheduler.step()` | 학습률 조정 |
| `running_loss += loss.item()` | 손실을 누적 (`.item()`은 텐서를 파이썬 숫자로 바꿈) |

- 에폭이 끝나면 `running_loss / len(train_loader)` = **평균 손실**을 출력합니다. 이 값이 줄어들면 학습이 잘 되고 있다는 뜻입니다.

**`test(model, test_loader)`**
- `model.eval()` : **평가 모드**로 전환
- `with torch.no_grad():` : 이 블록 안에서는 기울기를 계산하지 않음 (평가에는 필요 없으므로 더 빠르고 메모리 절약)
- `if isinstance(outputs, tuple): outputs = outputs[0]` : 뒤에서 만들 모델들은 값을 **2개 묶음(튜플)** 으로 돌려주는데, 그 경우 첫 번째(클래스 점수)만 사용합니다.
- `_, predicted = torch.max(outputs.data, 1)` : 이미지마다 10개 점수 중 **최댓값과 그 위치**를 돌려줍니다. 값은 필요 없어 `_`로 버리고, 위치(= 예측한 클래스)만 씁니다.
- `correct += (predicted == labels).sum().item()` : 맞힌 개수 누적
- `accuracy = 100 * correct / total` : 정확도(%)를 출력하고 돌려줌

## Load & Evaluate Teacher Model

Knowledge Distillation의 기준이 되는 **Teacher 모델(VGGCifar9)** 을 초기화하고, 앞서 불러온 pretrained 가중치를 적용합니다. 이후, CIFAR-10 test set에 대해 성능을 평가합니다.

In [7]:
# 7
teacher_model = VGGCifar9().cuda()
teacher_model.load_state_dict(state_dict)
test_accuracy_teacher = test(teacher_model, test_loader)

Test Accuracy: 92.95%


### 📘 셀 7 해설
**한 줄 요약:** **Teacher 모델**을 만들고, 미리 학습된 가중치를 넣은 뒤 정확도를 확인합니다.

- `teacher_model = VGGCifar9().cuda()` : Setup의 설계도로 모델을 만들어 GPU에 올림 (이 시점의 가중치는 무작위 값)
- `teacher_model.load_state_dict(state_dict)` : 내려받아 둔 **학습 완료 가중치**를 채워 넣음
- `test_accuracy_teacher = test(...)` : 정확도를 재서 변수에 저장

**출력 해석:** `92.95%` — Teacher는 이미 충분히 학습된 "선생님"입니다. Teacher는 이 노트북에서 **더 이상 학습하지 않고**, Student를 가르치는 데만 쓰입니다.

## 모델 초기화 일관성 확인

Knowledge Distillation 실험에서 **공정한 비교**를 위해 Student 모델의 초기화 상태가 동일한지 확인하는 과정입니다. 여기서는 동일한 구조를 가진 두 개의 `VGGCifar5` 모델을 생성하고, 첫 번째 convolution layer의 weight norm을 비교합니다.

In [8]:
# 8
# Print the norm of the first layer of the initial lightweight model
student_model = VGGCifar5().cuda()
print("Norm of 1st layer of student_model:", torch.norm(student_model.backbone[0].weight).item())

# Print the norm of the first layer of the new lightweight model
student_model_2 = VGGCifar5().cuda()
print("Norm of 1st layer of student_model_2:", torch.norm(student_model_2.backbone[0].weight).item())

Norm of 1st layer of student_model: 4.603283882141113
Norm of 1st layer of student_model_2: 4.603283882141113


### 📘 셀 8 해설
**한 줄 요약:** Student 모델을 두 번 만들어도 **시작 가중치가 똑같은지** 확인합니다.

- `student_model.backbone[0].weight` : backbone의 0번째 층(첫 번째 Conv)의 가중치
- `torch.norm(...)` : 텐서의 **크기(norm)** = 모든 값을 제곱해서 더한 뒤 제곱근을 취한 숫자 하나. 가중치가 같으면 이 값도 같으므로 간단한 확인 수단으로 씁니다.

**출력 해석:** 두 모델 모두 `4.603283882141113`으로 **완전히 같습니다.**

**왜 중요할까?** 신경망의 가중치는 처음에 무작위로 정해지는데, 시작점이 다르면 결과도 조금씩 달라집니다. Setup에서 `VGGCifar5`를 만들 때마다 `set_seed()`를 부르도록 해 두었기 때문에 항상 같은 값에서 출발합니다. 덕분에 뒤의 실험들에서 정확도 차이가 나면 "운"이 아니라 **학습 방법의 차이**라고 말할 수 있습니다.

## 모델 파라미터 수 비교

Teacher와 Student 모델 간의 **복잡도 차이**를 수치적으로 비교하기 위해 전체 파라미터 개수를 출력합니다. 이는 Knowledge Distillation의 핵심 가정인 "*성능은 높지만 무거운 Teacher → 가볍고 빠른 Student로 지식 이전*"을 정량적으로 뒷받침하는 자료가 됩니다.

In [9]:
# 9
total_params_teacher = "{:,}".format(sum(p.numel() for p in teacher_model.parameters()))
print(f"Teaher model parameters: {total_params_teacher}")
total_params_student = "{:,}".format(sum(p.numel() for p in student_model.parameters()))
print(f"Student model parameters: {total_params_student}")

Teaher model parameters: 9,228,362
Student model parameters: 964,170


### 📘 셀 9 해설
**한 줄 요약:** Teacher와 Student의 **파라미터(가중치) 개수**를 세어 비교합니다.

`"{:,}".format(sum(p.numel() for p in teacher_model.parameters()))` 을 안쪽부터 읽으면:
- `teacher_model.parameters()` : 모델의 모든 가중치 텐서를 하나씩 꺼냄
- `p.numel()` : 그 텐서 안의 숫자 개수
- `sum(... for p in ...)` : 전부 더함. 괄호 안에 `for`를 쓰는 이 형태를 **제너레이터 표현식**이라고 하며, 반복문을 한 줄로 쓴 것입니다.
- `"{:,}".format(숫자)` : 숫자에 **천 단위 쉼표**를 넣어 문자열로 만듦 (9228362 → `9,228,362`)

**출력 해석:** Teacher `9,228,362`개, Student `964,170`개 → Student는 Teacher의 **약 1/10 크기**입니다. 이렇게 작은 모델이 Teacher에 가까운 성능을 내게 만드는 것이 이 실습의 목표입니다.

> 출력의 `Teaher`는 `Teacher`의 오타입니다(문자열 안의 글자라 동작에는 영향 없음).

## Student 모델 단독 학습 (Cross-Entropy Only)

본 단계에서는 Student 모델을 **Teacher의 도움 없이** 단독으로 학습시킵니다. 이 실험은 이후 Knowledge Distillation을 적용했을 때 얼마나 성능이 향상되는지를 비교하기 위한 **Baseline** 성능을 확보하는 과정입니다.

In [10]:
# 10
student_model = VGGCifar5().cuda()
train(student_model, get_train_loader(train_dataset), epochs=5, learning_rate=0.01)
test_accuracy_student_ce = test(student_model, test_loader)

Epoch 1/5, Loss: 1.3420269849050381
Epoch 2/5, Loss: 0.8246572953660775
Epoch 3/5, Loss: 0.5881544247155299
Epoch 4/5, Loss: 0.39004730400831805
Epoch 5/5, Loss: 0.2583794688705898
Test Accuracy: 83.01%


### 📘 셀 10 해설
**한 줄 요약:** 비교 기준(**Baseline**)으로, Student를 **Teacher 도움 없이 혼자** 학습시킵니다.

- `student_model = VGGCifar5().cuda()` : 새 Student 생성 (항상 같은 초기값)
- `train(student_model, get_train_loader(train_dataset), epochs=5, learning_rate=0.01)` : 셀 6의 함수로 5 에폭 학습. 정답 라벨만 보고 배웁니다(Cross-Entropy만 사용).
- `test_accuracy_student_ce = test(...)` : 정확도 저장. 변수 이름의 `ce`는 Cross-Entropy를 뜻합니다.

**출력 해석**
- 손실: `1.34 → 0.82 → 0.59 → 0.39 → 0.26` 으로 꾸준히 감소
- 테스트 정확도: **`83.01%`** — 이 숫자가 뒤의 모든 KD 실험이 넘어야 할 기준선입니다.

## 정확도 결과 요약

In [11]:
# 11
print(f"Teacher accuracy: {test_accuracy_teacher:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_student_ce:.2f}%")

Teacher accuracy: 92.95%
Student accuracy without teacher: 83.01%


### 📘 셀 11 해설
**한 줄 요약:** 지금까지의 결과를 나란히 출력합니다.

- Teacher: `92.95%` / Student(혼자 학습): `83.01%` → 약 **10%p 차이**

파라미터가 1/10인 만큼 성능 차이가 납니다. 이 차이를 Teacher의 도움으로 얼마나 줄일 수 있는지가 이후 실험의 관심사입니다.

# 3.2. Knowledge Distillation (Soft Targets)

## [실습 1] Knowledge Distillation 학습 함수 정의

아래 함수는 **Teacher 모델의 soft output**을 활용하여 Student 모델을 지도 학습하는 Knowledge Distillation (KD) 학습 루프입니다.  
기존 Cross-Entropy 학습에 더해, soft target을 이용한 추가적인 loss를 도입하여 Student가 Teacher의 예측 구조까지 학습할 수 있도록 합니다.


### KD 핵심 개념

- **Soft Targets**: Teacher의 출력(logits)을 softmax로 부드럽게 만든 확률 분포
- **Temperature (T)**: softmax 분포의 평탄함을 제어하며, 높을수록 클래스 간 정보가 더 많이 보존됨
- **Loss 조합**:
  - `CrossEntropyLoss`: Ground-truth label 기반 지도 손실
  - `KL-like Loss`: Teacher의 soft target 분포와 Student 예측 분포 간 차이를 최소화하는 손실
  - 두 손실을 weighted sum으로 조합

In [12]:
# 12
def train_knowledge_distillation(teacher,
                                 student,
                                 train_loader,
                                 epochs,
                                 learning_rate,
                                 T,  # temperature
                                 soft_target_loss_weight,
                                 ce_loss_weight):
    ce_loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs * len(train_loader))

    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.cuda(), labels.cuda()

            optimizer.zero_grad()

            ##################### YOUR CODE STARTS HERE #####################
            # Forward pass with the teacher model - do not save gradients here as we do not change the teacher's weights
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            # Forward pass with the student model
            student_logits = student(inputs)

            # Soften the student logits by applying softmax
            # Hint: nn.functional.softmax()
            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            student_prob = nn.functional.softmax(student_logits / T, dim=-1)

            # Calculate the soft targets loss. Scaled by T**2 as suggested by the authors of the paper "Distilling the knowledge in a neural network"
            soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - student_prob.log())) / student_prob.size(0) * (T**2)

            # Calculate the true label loss
            label_loss = ce_loss(student_logits, labels)

            # Weighted sum of the two losses
            loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss
            ##################### YOUR CODE ENDS HERE #######################

            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

### 📘 셀 12 해설
**한 줄 요약:** [실습 1 정답] **Teacher의 "부드러운 답"을 Student가 따라 하도록** 학습시키는 Knowledge Distillation 함수입니다.

**핵심 아이디어:** 정답 라벨은 "이 사진은 고양이"만 알려 줍니다. 그런데 Teacher의 출력은 "고양이 70%, 개 25%, 트럭 0.1%..."처럼 **클래스 사이의 닮은 정도**까지 담고 있습니다. 이 추가 정보(soft target)를 Student에게 함께 가르칩니다.

**준비** : 셀 6과 같은 손실 함수·옵티마이저·스케줄러. 단, `optim.Adam(student.parameters(), ...)`으로 **Student의 가중치만** 학습합니다. `teacher.eval()` / `student.train()`으로 Teacher는 평가 모드, Student는 학습 모드로 둡니다.

**정답 부분 (묶음마다 실행)**

| 코드 | 의미 |
|---|---|
| `with torch.no_grad(): teacher_logits = teacher(inputs)` | Teacher의 점수 계산. Teacher는 학습하지 않으므로 기울기 계산을 끕니다. |
| `student_logits = student(inputs)` | Student의 점수 계산 |
| `soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)` | Teacher 점수를 `T`로 나눈 뒤 확률로 변환 |
| `student_prob = nn.functional.softmax(student_logits / T, dim=-1)` | Student도 같은 방식으로 확률로 변환 |
| `soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - student_prob.log())) / student_prob.size(0) * (T**2)` | 두 확률 분포의 **차이** |
| `label_loss = ce_loss(student_logits, labels)` | 정답 라벨과의 차이 (기존 방식) |
| `loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss` | 두 손실을 **가중치를 곱해 더함** |

**Temperature `T`의 역할** — 점수를 `T`로 나누면 확률이 **평평해집니다.**
- 점수가 [5, 2, 0]일 때, T=1이면 확률이 약 [0.94, 0.05, 0.01]로 한쪽에 쏠립니다.
- T=10이면 약 [0.41, 0.31, 0.28]로 고르게 퍼져, 2등·3등 클래스의 정보가 잘 드러납니다.

**`soft_targets_loss` 식 읽기**
- `soft_targets * (soft_targets.log() - student_prob.log())` : 두 확률 분포의 차이를 재는 **KL divergence** 공식입니다. 두 분포가 같으면 0이 됩니다.
- `torch.sum(...) / student_prob.size(0)` : 전부 더한 뒤 묶음 크기(`size(0)`)로 나눔 → 이미지 1장당 평균
- `* (T**2)` : `T`로 나누면 기울기가 그만큼 작아지므로, 논문에서 제안한 대로 T²을 곱해 크기를 보정합니다.

`dim=-1`은 "마지막 차원(클래스 방향)에 대해" 계산하라는 뜻입니다.

> 코드 주석에는 "Soften the student logits"라고만 적혀 있지만, 실제로는 Teacher와 Student **양쪽 모두** 같은 `T`로 부드럽게 만듭니다.

## Knowledge Distillation 학습 수행

앞서 정의한 `train_knowledge_distillation()` 함수를 이용하여, Teacher 모델의 soft prediction을 기반으로 Student 모델을 학습시킵니다. 이후, 테스트 정확도를 측정하여 **기존 CE-only 학습과의 성능 차이**를 비교합니다.

In [13]:
# 13
student_model = VGGCifar5().cuda()
train_knowledge_distillation(teacher=teacher_model,
                             student=student_model,
                             train_loader=get_train_loader(train_dataset),
                             epochs=5,
                             learning_rate=0.01,
                             T=10,
                             soft_target_loss_weight=0.5,
                             ce_loss_weight=0.5)

test_accuracy_student_ce_and_kd = test(student_model, test_loader)

Epoch 1/5, Loss: 7.034169396476063
Epoch 2/5, Loss: 3.49262972865873
Epoch 3/5, Loss: 2.2329331636428833
Epoch 4/5, Loss: 1.52623591398644
Epoch 5/5, Loss: 1.1698705320772917
Test Accuracy: 85.53%


### 📘 셀 13 해설
**한 줄 요약:** 방금 만든 함수로 Student를 **KD 방식으로 학습**시키고 정확도를 잽니다.

- `student_model = VGGCifar5().cuda()` : **새** Student로 다시 시작 (셀 10과 같은 초기값)
- `teacher=teacher_model, student=student_model, ...` : 인자를 `이름=값` 형태로 넘기는 **키워드 인자**. 순서를 헷갈릴 일이 없고 읽기 쉽습니다.
- `T=10` : Temperature / `soft_target_loss_weight=0.5`, `ce_loss_weight=0.5` : 두 손실을 반반씩 반영
- 에폭 수(5)와 학습률(0.01)은 셀 10과 같게 두어 공정하게 비교합니다.

**출력 해석**
- 손실: `7.03 → 3.49 → 2.23 → 1.53 → 1.17`. 셀 10보다 숫자가 큰 것은 **손실의 정의가 다르기 때문**(T² = 100이 곱해짐)이며, 성능이 나쁘다는 뜻이 아닙니다. 손실 값은 같은 실험 안에서 줄어드는지만 보면 됩니다.
- 테스트 정확도: **`85.53%`** (혼자 학습했을 때 83.01%)

## 정확도 결과 요약

In [14]:
# 14
print(f"Teacher accuracy: {test_accuracy_teacher:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_student_ce:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_student_ce_and_kd:.2f}%")

Teacher accuracy: 92.95%
Student accuracy without teacher: 83.01%
Student accuracy with CE + KD: 85.53%


### 📘 셀 14 해설
**한 줄 요약:** KD 결과를 추가해 비교합니다.

| 모델 | 정확도 |
|---|---|
| Teacher | 92.95% |
| Student (혼자 학습) | 83.01% |
| Student (CE + KD) | **85.53%** |

모델 구조·초기값·에폭 수·학습 데이터가 모두 같은데, **Teacher의 soft target을 함께 배운 것만으로 +2.52%p** 올랐습니다. 추론할 때의 Student는 크기도 속도도 그대로이므로, 추가 비용 없이 얻은 향상입니다.

# 3.3. Cosine Loss Minimization (Cosine Loss)

## Cosine Similarity 기반 KD 모델 정의

본 실험에서는 **Teacher와 Student 모델의 내부 표현(hidden representation)** 을 정렬하여 학습 효과를 높이고자 합니다.  
이를 위해 기존 VGG 구조를 변형하여 **flatten된 feature representation을 반환하는** 모델을 정의합니다.

In [15]:
# 15
class VGGCifar9_Cosine(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.backbone = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, 64, 3, padding=1, bias=False)),
            ('bn0', nn.BatchNorm2d(64)),
            ('relu0', nn.ReLU(True)),
            ('conv1', nn.Conv2d(64, 128, 3, padding=1, bias=False)),
            ('bn1', nn.BatchNorm2d(128)),
            ('relu1', nn.ReLU(True)),
            ('pool0', nn.MaxPool2d(2)),
            ('conv2', nn.Conv2d(128, 256, 3, padding=1, bias=False)),
            ('bn2', nn.BatchNorm2d(256)),
            ('relu2', nn.ReLU(True)),
            ('conv3', nn.Conv2d(256, 256, 3, padding=1, bias=False)),
            ('bn3', nn.BatchNorm2d(256)),
            ('relu3', nn.ReLU(True)),
            ('pool1', nn.MaxPool2d(2)),
            ('conv4', nn.Conv2d(256, 512, 3, padding=1, bias=False)),
            ('bn4', nn.BatchNorm2d(512)),
            ('relu4', nn.ReLU(True)),
            ('conv5', nn.Conv2d(512, 512, 3, padding=1, bias=False)),
            ('bn5', nn.BatchNorm2d(512)),
            ('relu5', nn.ReLU(True)),
            ('pool2', nn.MaxPool2d(2)),
            ('conv6', nn.Conv2d(512, 512, 3, padding=1, bias=False)),
            ('bn6', nn.BatchNorm2d(512)),
            ('relu6', nn.ReLU(True)),
            ('conv7', nn.Conv2d(512, 512, 3, padding=1, bias=False)),
            ('bn7', nn.BatchNorm2d(512)),
            ('relu7', nn.ReLU(True)),
            ('pool3', nn.MaxPool2d(2)),
        ]))
        self.classifier = nn.Linear(512, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        conv_output = torch.flatten(x, 1)
        conv_output_after_pooling = torch.nn.functional.avg_pool1d(conv_output, 2)
        x = x.mean([2, 3])
        x = self.classifier(x)
        return x, conv_output_after_pooling

class VGGCifar5_Cosine(nn.Module):
    def __init__(self) -> None:
        # Generate the same scratch model
        set_seed()
        super().__init__()
        self.backbone = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, 64, 3, padding=1, bias=False)),
            ('bn0', nn.BatchNorm2d(64)),
            ('relu0', nn.ReLU(True)),
            ('pool0', nn.MaxPool2d(2)),
            ('conv1', nn.Conv2d(64, 128, 3, padding=1, bias=False)),
            ('bn1', nn.BatchNorm2d(128)),
            ('relu1', nn.ReLU(True)),
            ('pool1', nn.MaxPool2d(2)),
            ('conv2', nn.Conv2d(128, 256, 3, padding=1, bias=False)),
            ('bn2', nn.BatchNorm2d(256)),
            ('relu2', nn.ReLU(True)),
            ('pool2', nn.MaxPool2d(2)),
            ('conv3', nn.Conv2d(256, 256, 3, padding=1, bias=False)),
            ('bn3', nn.BatchNorm2d(256)),
            ('relu3', nn.ReLU(True)),
            ('pool3', nn.MaxPool2d(2)),
        ]))
        self.classifier = nn.Linear(256, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        conv_output = torch.flatten(x, 1)
        x = x.mean([2, 3])
        # conv_output = x
        x = self.classifier(x)
        return x, conv_output

### 📘 셀 15 해설
**한 줄 요약:** 이번에는 최종 답이 아니라 **모델 내부의 중간 표현(feature)** 을 비교하기 위해, **값을 2개 돌려주는** Teacher / Student 모델을 새로 정의합니다.

구조(backbone, classifier)는 Setup의 `VGGCifar9`, `VGGCifar5`와 **완전히 같고**, `forward`만 다릅니다.

**Teacher (`VGGCifar9_Cosine`)의 `forward`**
- `x = self.backbone(x)` : 합성곱 부분 통과 → 모양 (묶음, 512, 2, 2)
- `conv_output = torch.flatten(x, 1)` : 1번 차원부터 끝까지를 **한 줄로 폄** → (묶음, 2048)
- `torch.nn.functional.avg_pool1d(conv_output, 2)` : 이웃한 값 **2개씩 평균** → 길이가 절반인 (묶음, 1024)
- `x = x.mean([2, 3])` : 2×2 위치에 대해 평균 → (묶음, 512)
- `x = self.classifier(x)` : 클래스 10개의 점수
- `return x, conv_output_after_pooling` : **(점수, 중간 표현)** 2개를 돌려줌

**Student (`VGGCifar5_Cosine`)의 `forward`**
- backbone 출력이 (묶음, 256, 2, 2)이므로 펴면 바로 (묶음, 1024) → 평균 풀링 없이 그대로 돌려줍니다.

**Teacher 쪽만 절반으로 줄이는 이유:** 두 모델의 중간 표현을 비교하려면 **길이가 같아야** 합니다. Teacher는 2048, Student는 1024이므로 Teacher 쪽을 1024로 맞춘 것입니다.

> `nn.Sequential(OrderedDict([('conv0', ...), ...]))` : 층들을 **이름을 붙여** 순서대로 쌓는 방법입니다. `set_seed()`는 Student가 항상 같은 초기값을 갖게 합니다.

## Cosine Loss 기반 KD를 위한 모델 초기화

앞서 정의한 **representation-level KD 구조**를 활용하여 Teacher와 Student 모델을 초기화합니다.  
Teacher는 기존 `VGGCifar9`의 학습된 가중치를 그대로 활용하며, Student는 새로 초기화하여 학습을 수행할 준비를 합니다.


In [16]:
# 16
teacher_model_cosine = VGGCifar9_Cosine().cuda()
teacher_model_cosine.load_state_dict(state_dict)

student_model_cosine = VGGCifar5_Cosine().cuda()

### 📘 셀 16 해설
**한 줄 요약:** 새 구조의 Teacher와 Student를 만듭니다.

- `teacher_model_cosine.load_state_dict(state_dict)` : 층 구성이 원래 Teacher와 같으므로 **같은 학습 완료 가중치**를 그대로 넣을 수 있습니다. (바뀐 것은 `forward`의 반환값뿐)
- `student_model_cosine = VGGCifar5_Cosine().cuda()` : Student는 학습 전 상태로 생성

## Cosine Distillation을 위한 Representation 차원 확인

CosineEmbeddingLoss를 적용하기 위해서는 Teacher와 Student 모델이 반환하는 **hidden representation**이 동일한 차원을 가져야 합니다.  
아래 코드는 임의의 입력(batch) 데이터를 각각의 모델에 통과시켜, 출력되는 **logits**과 **flatten된 convolutional feature vector**의 shape을 출력합니다.

In [17]:
# 17
# Create a sample input tensor
sample_input = torch.randn(128, 3, 32, 32).cuda() # Batch size: 128, Filters: 3, Image size: 32x32

# Pass the input through the student
logits, hidden_representation = student_model_cosine(sample_input)

# Print the shapes of the tensors
print("Student logits shape:", logits.shape) # batch_size x total_classes
print("Student hidden representation shape:", hidden_representation.shape) # batch_size x hidden_representation_size

# Pass the input through the teacher
logits, hidden_representation = teacher_model_cosine(sample_input)

# Print the shapes of the tensors
print("Teacher logits shape:", logits.shape) # batch_size x total_classes
print("Teacher hidden representation shape:", hidden_representation.shape) # batch_size x hidden_representation_size

Student logits shape: torch.Size([128, 10])
Student hidden representation shape: torch.Size([128, 1024])
Teacher logits shape: torch.Size([128, 10])
Teacher hidden representation shape: torch.Size([128, 1024])


### 📘 셀 17 해설
**한 줄 요약:** 가짜 입력을 넣어, 두 모델이 돌려주는 값의 **모양(shape)이 서로 맞는지** 확인합니다.

- `torch.randn(128, 3, 32, 32).cuda()` : 난수로 채운 가짜 이미지 128장 (3채널, 32×32)
- `logits, hidden_representation = student_model_cosine(sample_input)` : 모델이 돌려주는 2개 값을 변수 2개에 나눠 받음

**출력 해석**

| | logits | hidden representation |
|---|---|---|
| Student | (128, 10) | (128, **1024**) |
| Teacher | (128, 10) | (128, **1024**) |

중간 표현의 길이가 둘 다 1024로 같으므로, 다음 셀에서 두 벡터를 직접 비교할 수 있습니다.

## [실습 2] Cosine Similarity 기반 KD 학습 함수 정의

이 함수는 **Teacher와 Student의 내부 표현(hidden representation)** 간 유사도를 **CosineEmbeddingLoss**를 통해 극대화하는 방식으로 Student를 학습시킵니다.  
이는 soft-label 기반 KD와 달리 **representation-level distillation**으로 분류되며, Student의 feature extractor 품질 향상에 초점을 둡니다.

In [19]:
# 18
def train_cosine_loss(teacher,
                      student,
                      train_loader,
                      epochs,
                      learning_rate,
                      hidden_rep_loss_weight,
                      ce_loss_weight):
    ce_loss = nn.CrossEntropyLoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs * len(train_loader))

    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.cuda(), labels.cuda()

            optimizer.zero_grad()

            ##################### YOUR CODE STARTS HERE #####################
            # Forward pass with the teacher model and keep only the hidden representation
            with torch.no_grad():
                _, teacher_hidden_representation = teacher(inputs)

            # Forward pass with the student model
            student_logits, student_hidden_representation = student(inputs)

            # Calculate the cosine loss. Target is a vector of ones. From the loss formula above we can see that is
            # the case where loss minimization leads to cosine similarity increase.
            # Hint: cosine_loss(x, y, target)에서 target은 1로 이루어진 vector이며, torch.ones(inputs.size(0)).cuda())를 사용
            hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation,
                                          target=torch.ones(inputs.size(0)).cuda())

            # Calculate the true label loss
            label_loss = ce_loss(student_logits, labels)

            # Weighted sum of the two losses
            loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss
            ##################### YOUR CODE ENDS HERE #######################

            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

### 📘 셀 18 해설
**한 줄 요약:** [실습 2 정답] Student의 **중간 표현이 Teacher의 중간 표현과 같은 방향을 가리키도록** 학습시키는 함수입니다.

셀 12와 뼈대가 같고, 두 번째 손실만 다릅니다.

- `cosine_loss = nn.CosineEmbeddingLoss()` : 두 벡터의 **코사인 유사도**를 이용한 손실 함수

**정답 부분**

| 코드 | 의미 |
|---|---|
| `_, teacher_hidden_representation = teacher(inputs)` | Teacher의 반환값 2개 중 점수는 버리고(`_`) **중간 표현만** 사용 |
| `student_logits, student_hidden_representation = student(inputs)` | Student는 점수와 중간 표현 모두 사용 |
| `hidden_rep_loss = cosine_loss(student_..., teacher_..., target=torch.ones(inputs.size(0)).cuda())` | 두 중간 표현의 방향 차이 |
| `label_loss = ce_loss(student_logits, labels)` | 정답 라벨과의 차이 |
| `loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss` | 가중치를 곱해 더함 |

**코사인 유사도란?** 두 벡터가 이루는 **각도**로 닮은 정도를 재는 값입니다. 같은 방향이면 1, 직각이면 0, 반대 방향이면 -1입니다. 벡터의 길이(크기)는 상관하지 않고 방향만 봅니다.

**`target`의 의미:** `torch.ones(묶음 크기)`는 1로만 채운 벡터입니다. target이 1이면 "이 두 벡터는 **닮아야 하는 쌍**"이라는 뜻이고, 이때 손실은 `1 - 코사인 유사도`가 됩니다. 즉 손실을 줄이면 → 유사도가 1에 가까워지고 → Student의 내부 표현이 Teacher를 닮아 갑니다.

## Cosine Similarity 기반 Knowledge Distillation 실험

이 실험에서는 Teacher와 Student의 **internal feature vector** 간 유사도를 기반으로 하는 **CosineEmbeddingLoss**를 적용하여 Student 모델을 학습시킵니다.  
이 방법은 soft label이 아닌, feature-level에서의 표현력 유사성을 유도하는 방식으로 distillation을 수행합니다.

In [20]:
# 19
# Train and test the lightweight network with cross entropy loss
student_model_cosine = VGGCifar5_Cosine().cuda()
train_cosine_loss(teacher=teacher_model_cosine,
                  student=student_model_cosine,
                  train_loader=get_train_loader(train_dataset),
                  epochs=5,
                  learning_rate=0.01,
                  hidden_rep_loss_weight=0.5,
                  ce_loss_weight=0.5)

test_accuracy_student_ce_and_cosine_loss = test(student_model_cosine, test_loader)

Epoch 1/5, Loss: 0.8553633609086352
Epoch 2/5, Loss: 0.549352215531537
Epoch 3/5, Loss: 0.4127756560702458
Epoch 4/5, Loss: 0.3038669528863619
Epoch 5/5, Loss: 0.23534520946042922
Test Accuracy: 83.24%


### 📘 셀 19 해설
**한 줄 요약:** Cosine Loss 방식으로 Student를 학습시키고 정확도를 잽니다.

- `student_model_cosine = VGGCifar5_Cosine().cuda()` : 새 Student로 시작
- `hidden_rep_loss_weight=0.5, ce_loss_weight=0.5` : 두 손실을 반반씩 반영
- `test(...)` 안에서는 모델이 돌려준 튜플 중 첫 번째(점수)만 사용합니다(셀 6의 `isinstance(outputs, tuple)` 처리).

**출력 해석**
- 손실: `0.86 → 0.55 → 0.41 → 0.30 → 0.24` 로 감소
- 테스트 정확도: **`83.24%`**

> 첫 줄의 주석 `# Train and test the lightweight network with cross entropy loss`는 이전 코드에서 남은 것으로, 실제로는 Cross-Entropy와 Cosine Loss를 함께 사용합니다.

## 정확도 결과 요약

In [21]:
# 20
print(f"Teacher accuracy: {test_accuracy_teacher:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_student_ce:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_student_ce_and_kd:.2f}%")
print(f"Student accuracy with CE + CosineLoss: {test_accuracy_student_ce_and_cosine_loss:.2f}%")

Teacher accuracy: 92.95%
Student accuracy without teacher: 83.01%
Student accuracy with CE + KD: 85.53%
Student accuracy with CE + CosineLoss: 83.24%


### 📘 셀 20 해설
**한 줄 요약:** Cosine Loss 결과를 추가해 비교합니다.

| 모델 | 정확도 | 기준선 대비 |
|---|---|---|
| Teacher | 92.95% | |
| Student (혼자 학습) | 83.01% | 기준 |
| Student (CE + KD) | 85.53% | +2.52%p |
| Student (CE + CosineLoss) | 83.24% | +0.23%p |

이 실험 조건에서는 Cosine Loss의 효과(+0.23%p)가 **작았습니다.** 이 정도 차이는 설정에 따라 달라질 수 있는 수준입니다. 한 가지 이유로, Teacher(512채널)와 Student(256채널)의 중간 표현은 원래 서로 다른 것인데 길이만 맞춰 **직접** 비교했다는 점을 들 수 있습니다. 다음 실험은 이 사이에 "변환기"를 하나 두는 방법입니다.

# 3.4. Intermediate Regressor (Regressor + MSE)

## Feature Map Shape 비교

Hint-based Knowledge Distillation에서는 Teacher와 Student의 **중간 feature map**을 정렬(MSE 등)하기 위해,  
서로의 **convolutional output shape**을 일치시키거나 맞춰주는 작업이 필요합니다.  
아래 코드는 이 과정을 준비하기 위해 각 모델의 **backbone 출력 형태**를 비교합니다.

In [ ]:
# 21
# Pass the sample input only from the convolutional feature extractor
convolutional_fe_output_student = student_model.backbone(sample_input)
convolutional_fe_output_teacher = teacher_model.backbone(sample_input)

# Print their shapes
print("Student's feature extractor output shape: ", convolutional_fe_output_student.shape)
print("Teacher's feature extractor output shape: ", convolutional_fe_output_teacher.shape)

Student's feature extractor output shape:  torch.Size([128, 256, 2, 2])
Teacher's feature extractor output shape:  torch.Size([128, 512, 2, 2])


### 📘 셀 21 해설
**한 줄 요약:** Teacher와 Student의 **합성곱 부분(backbone) 출력 모양**을 비교합니다.

- `student_model.backbone(sample_input)` : 모델 전체가 아니라 **backbone 부분만** 따로 실행
- `.shape` : 결과 텐서의 모양

**출력 해석**
- Student: `(128, 256, 2, 2)` → 채널 **256개**
- Teacher: `(128, 512, 2, 2)` → 채널 **512개**

가로·세로 크기(2×2)는 같지만 **채널 수가 달라서** 두 feature map을 그대로는 비교할 수 없습니다. 그래서 다음 셀에서 Student 쪽에 256채널 → 512채널로 바꿔 주는 층을 추가합니다.

## Hint-based KD를 위한 Regressor 포함 모델 정의

이 실험에서는 Teacher의 중간 feature map과 Student의 feature map을 직접 정렬하기 위해 **trainable regressor**를 도입합니다.  
Teacher와 Student의 convolution output은 채널 수가 다르기 때문에, Student의 feature map을 **regressor**를 통해 변환하여 동일한 차원으로 맞춥니다.

In [22]:
# 22
class VGGCifar9_Regressor(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.backbone = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, 64, 3, padding=1, bias=False)),
            ('bn0', nn.BatchNorm2d(64)),
            ('relu0', nn.ReLU(True)),
            ('conv1', nn.Conv2d(64, 128, 3, padding=1, bias=False)),
            ('bn1', nn.BatchNorm2d(128)),
            ('relu1', nn.ReLU(True)),
            ('pool0', nn.MaxPool2d(2)),
            ('conv2', nn.Conv2d(128, 256, 3, padding=1, bias=False)),
            ('bn2', nn.BatchNorm2d(256)),
            ('relu2', nn.ReLU(True)),
            ('conv3', nn.Conv2d(256, 256, 3, padding=1, bias=False)),
            ('bn3', nn.BatchNorm2d(256)),
            ('relu3', nn.ReLU(True)),
            ('pool1', nn.MaxPool2d(2)),
            ('conv4', nn.Conv2d(256, 512, 3, padding=1, bias=False)),
            ('bn4', nn.BatchNorm2d(512)),
            ('relu4', nn.ReLU(True)),
            ('conv5', nn.Conv2d(512, 512, 3, padding=1, bias=False)),
            ('bn5', nn.BatchNorm2d(512)),
            ('relu5', nn.ReLU(True)),
            ('pool2', nn.MaxPool2d(2)),
            ('conv6', nn.Conv2d(512, 512, 3, padding=1, bias=False)),
            ('bn6', nn.BatchNorm2d(512)),
            ('relu6', nn.ReLU(True)),
            ('conv7', nn.Conv2d(512, 512, 3, padding=1, bias=False)),
            ('bn7', nn.BatchNorm2d(512)),
            ('relu7', nn.ReLU(True)),
            ('pool3', nn.MaxPool2d(2)),
        ]))
        self.classifier = nn.Linear(512, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        conv_feature_map = x
        x = x.mean([2, 3])
        x = self.classifier(x)
        return x, conv_feature_map

class VGGCifar5_Regressor(nn.Module):
    def __init__(self) -> None:
        # Generate the same scratch model
        set_seed()
        super().__init__()
        self.backbone = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, 64, 3, padding=1, bias=False)),
            ('bn0', nn.BatchNorm2d(64)),
            ('relu0', nn.ReLU(True)),
            ('pool0', nn.MaxPool2d(2)),
            ('conv1', nn.Conv2d(64, 128, 3, padding=1, bias=False)),
            ('bn1', nn.BatchNorm2d(128)),
            ('relu1', nn.ReLU(True)),
            ('pool1', nn.MaxPool2d(2)),
            ('conv2', nn.Conv2d(128, 256, 3, padding=1, bias=False)),
            ('bn2', nn.BatchNorm2d(256)),
            ('relu2', nn.ReLU(True)),
            ('pool2', nn.MaxPool2d(2)),
            ('conv3', nn.Conv2d(256, 256, 3, padding=1, bias=False)),
            ('bn3', nn.BatchNorm2d(256)),
            ('relu3', nn.ReLU(True)),
            ('pool3', nn.MaxPool2d(2)),
        ]))
        self.regressor = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512)
        )
        self.classifier = nn.Linear(256, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        regressor_output = self.regressor(x)
        x = x.mean([2, 3])
        x = self.classifier(x)
        return x, regressor_output

### 📘 셀 22 해설
**한 줄 요약:** feature map 자체를 비교하기 위한 Teacher / Student 모델을 정의합니다. Student에는 채널 수를 맞춰 주는 **regressor(변환 층)** 가 추가됩니다.

**Teacher (`VGGCifar9_Regressor`)**
- `conv_feature_map = x` : backbone 출력(묶음, 512, 2, 2)을 **펴지 않고 그대로** 보관
- `return x, conv_feature_map` : (점수, feature map)을 돌려줌

**Student (`VGGCifar5_Regressor`)**
- 새로 추가된 부분:
```python
self.regressor = nn.Sequential(
    nn.Conv2d(256, 512, 3, padding=1, bias=False),   # 256채널 → 512채널
    nn.BatchNorm2d(512)
)
```
- `regressor_output = self.regressor(x)` : backbone 출력(256채널)을 512채널로 변환 → Teacher의 feature map과 **같은 모양**
- `x = x.mean([2, 3])` → `self.classifier(x)` : 분류는 **regressor를 거치지 않은 원래 `x`** 로 수행
- `return x, regressor_output` : (점수, 변환된 feature map)

**중요한 점:** regressor는 **학습할 때 Teacher와 비교하는 용도로만** 쓰입니다. 분류 결과를 내는 경로에는 들어가지 않으므로, 학습이 끝나면 없어도 되는 "보조 도구"입니다.

## Hint-based KD용 Teacher 모델 초기화 및 가중치 로딩

Hint-based Knowledge Distillation에서는 Teacher 모델의 중간 feature map을 **지도 정보로 활용**합니다.  
이를 위해 기존 학습된 `VGGCifar9`의 가중치를 기반으로 하되, **feature map 반환이 가능한 구조**로 변경한 `VGGCifar9_Regressor` 모델을 사용합니다.

In [23]:
# 23
teacher_model_reg = VGGCifar9_Regressor().cuda()
teacher_model_reg.load_state_dict(state_dict)

<All keys matched successfully>

### 📘 셀 23 해설
**한 줄 요약:** 새 구조의 Teacher를 만들고 학습 완료 가중치를 넣습니다.

**출력 해석:** `<All keys matched successfully>` — `load_state_dict`가 돌려준 메시지로, 저장된 가중치의 이름과 모델의 층 이름이 **전부 일치해서 문제없이 들어갔다**는 뜻입니다. (주피터는 셀 마지막 줄의 결과 값을 자동으로 출력합니다. 셀 16에서는 뒤에 다른 줄이 있어서 출력되지 않았습니다.)

## [실습 3] Hint-based Knowledge Distillation 학습 함수 정의 (MSE Loss 기반)

이 함수는 **중간 feature map**을 기준으로 Teacher와 Student의 표현을 정렬하기 위해 **Mean Squared Error (MSE) Loss**를 사용하는 Hint-based KD 학습 방식입니다.  
이를 통해 Student의 feature extractor가 Teacher의 중간 표현력을 모방하도록 유도합니다.


**학습 개념 요약**

| 손실 종류      | 역할                                       |
|----------------|--------------------------------------------|
| CrossEntropy   | 정답 라벨 기반 분류 학습                   |
| MSE Loss       | Teacher feature map ↔ Student regressed feature map 간 오차 최소화 |

In [25]:
# 24
def train_mse_loss(teacher,
                   student,
                   train_loader,
                   epochs,
                   learning_rate,
                   feature_map_weight,
                   ce_loss_weight):
    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs * len(train_loader))

    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.cuda(), labels.cuda()

            optimizer.zero_grad()

            ##################### YOUR CODE STARTS HERE #####################
            # Again ignore teacher logits
            with torch.no_grad():
                _, teacher_feature_map = teacher(inputs)

            # Forward pass with the student model
            student_logits, regressor_feature_map = student(inputs)

            # Calculate the loss
            hidden_rep_loss = mse_loss(regressor_feature_map, teacher_feature_map)

            # Calculate the true label loss
            label_loss = ce_loss(student_logits, labels)

            # Weighted sum of the two losses
            loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss
            ##################### YOUR CODE ENDS HERE #######################

            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

### 📘 셀 24 해설
**한 줄 요약:** [실습 3 정답] Student의 (변환된) feature map이 **Teacher의 feature map과 값 자체가 비슷해지도록** 학습시키는 함수입니다.

역시 셀 12, 18과 뼈대가 같고 두 번째 손실만 다릅니다.

- `mse_loss = nn.MSELoss()` : **평균 제곱 오차**(Mean Squared Error). 같은 위치의 값끼리 빼서 제곱한 뒤 평균을 냅니다. 두 텐서가 완전히 같으면 0입니다.

**정답 부분**

| 코드 | 의미 |
|---|---|
| `_, teacher_feature_map = teacher(inputs)` | Teacher의 feature map만 사용 (기울기 계산 끔) |
| `student_logits, regressor_feature_map = student(inputs)` | Student의 점수와 regressor 출력 |
| `hidden_rep_loss = mse_loss(regressor_feature_map, teacher_feature_map)` | 두 feature map의 차이 |
| `label_loss = ce_loss(student_logits, labels)` | 정답 라벨과의 차이 |
| `loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss` | 가중치를 곱해 더함 |

**세 가지 방식 비교**

| 방식 | Teacher에게서 배우는 것 | 손실 함수 |
|---|---|---|
| Soft Targets (셀 12) | 최종 출력의 확률 분포 | KL divergence |
| Cosine Loss (셀 18) | 중간 표현의 **방향** | CosineEmbeddingLoss |
| Regressor + MSE (이 셀) | 중간 feature map의 **값 자체** | MSELoss |

`optim.Adam(student.parameters(), ...)`에는 regressor의 가중치도 포함되므로, **regressor도 함께 학습**됩니다.

## Hint-based KD (Regressor + MSE Loss) 실험

이 실험에서는 Student가 Teacher의 **중간 feature map**을 직접 모방하도록 유도하는 **Hint-based Knowledge Distillation**을 수행합니다.  
이를 위해 Student에 **trainable regressor layer**를 도입하고, Teacher의 표현을 **MSE Loss**로 정렬합니다.

In [26]:
# 25
student_model_reg = VGGCifar5_Regressor().cuda()
train_mse_loss(teacher=teacher_model_reg,
               student=student_model_reg,
               train_loader=get_train_loader(train_dataset),
               epochs=5,
               learning_rate=0.01,
               feature_map_weight=0.5,
               ce_loss_weight=0.5)

test_accuracy_student_ce_and_mse_loss = test(student_model_reg, test_loader)

KeyboardInterrupt: 

### 📘 셀 25 해설
**한 줄 요약:** Regressor + MSE 방식으로 Student를 학습시키고 정확도를 재는 셀입니다. 구성은 셀 13, 19와 같습니다.

- `student_model_reg = VGGCifar5_Regressor().cuda()` : regressor가 달린 새 Student
- `feature_map_weight=0.5, ce_loss_weight=0.5` : 두 손실을 반반씩 반영

**출력 해석:** 저장된 노트북에는 **`KeyboardInterrupt`** 만 남아 있습니다. 이는 오류가 아니라 실행 도중 **사용자가 직접 중단**(정지 버튼)했다는 표시입니다. 따라서 이 파일에는 이 실험의 손실 값과 정확도가 **기록되어 있지 않습니다.** 셀을 끝까지 실행하면 다른 실험과 같은 형식으로 에폭별 손실과 `Test Accuracy`가 출력됩니다.

## 정확도 결과 요약

In [ ]:
# 26
print(f"Teacher accuracy: {test_accuracy_teacher:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_student_ce:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_student_ce_and_kd:.2f}%")
print(f"Student accuracy with CE + CosineLoss: {test_accuracy_student_ce_and_cosine_loss:.2f}%")
print(f"Student accuracy with CE + RegressorMSE: {test_accuracy_student_ce_and_mse_loss:.2f}%")

### 📘 셀 26 해설
**한 줄 요약:** 네 가지 Student 학습 결과를 Teacher와 함께 한꺼번에 출력하는 최종 요약 셀입니다.

**출력이 없는 이유:** 바로 앞 셀 25가 중단되어 `test_accuracy_student_ce_and_mse_loss` 변수가 만들어지지 않았기 때문입니다. 이 상태로 실행하면 "그런 이름의 변수가 없다"는 `NameError`가 납니다. **셀 25를 끝까지 실행한 뒤** 이 셀을 실행하면 5줄이 모두 출력됩니다.

**저장된 결과로 본 정리**

| 모델 | 정확도 |
|---|---|
| Teacher (파라미터 약 923만 개) | 92.95% |
| Student 혼자 학습 (약 96만 개) | 83.01% |
| Student + KD (Soft Targets) | 85.53% |
| Student + Cosine Loss | 83.24% |
| Student + Regressor MSE | (실행 결과 없음) |

**전체 실습 정리**
1. 작은 모델을 정답 라벨만으로 학습시키는 것보다, **큰 모델의 출력을 함께 따라 하게 하면** 같은 크기로 더 높은 정확도를 얻을 수 있습니다.
2. Teacher에게서 배울 수 있는 것은 최종 출력(soft target)뿐 아니라 **중간 표현**(방향, feature map 값)도 있습니다.
3. 세 방식 모두 "**정답 손실 + Teacher를 따라 하는 손실**을 가중치로 더한다"는 같은 틀을 사용하며, `T`와 손실 가중치는 조절할 수 있는 값입니다.